# Implementación de CART (Classification And Regression Tree)

Este notebook implementa el algoritmo CART sobre el conjunto de datos de reviews de videojuegos. Se realizará un análisis exhaustivo con diferentes hiperparámetros y se evaluará su rendimiento.

## Objetivos
1. Implementar CART con al menos 3 hiperparámetros
2. Evaluar el modelo con diferentes combinaciones de hiperparámetros
3. Comparar rendimiento en conjuntos de entrenamiento y validación
4. Evaluar el modelo final en datos de prueba
5. Realizar predicciones con un nuevo dato (inventado)
6. Analizar resultados y conclusiones

In [ ]:
# Importar bibliotecas necesarias
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
import seaborn as sns

: 

In [ ]:
# Cargar los datos de entrenamiento, validación y prueba
train_X = pd.read_csv('../Data_Prep/TrainX.csv')
train_y = pd.read_csv('../Data_Prep/TrainY.csv')
val_X = pd.read_csv('../Data_Prep/ValidationX.csv')
val_y = pd.read_csv('../Data_Prep/ValidationY.csv')
test_X = pd.read_csv('../Data_Prep/TestX.csv')
test_y = pd.read_csv('../Data_Prep/TestY.csv')

print("Dimensiones de los conjuntos de datos:")
print(f"Entrenamiento: {train_X.shape}")
print(f"Validación: {val_X.shape}")
print(f"Prueba: {test_X.shape}")

## Entrenamiento del modelo con diferentes hiperparámetros

Se utilizaran los siguientes hiperparámetros para el árbol de decisión:
1. `max_depth`: Profundidad máxima del árbol
2. `min_samples_split`: Número mínimo de muestras requeridas para dividir un nodo
3. `min_samples_leaf`: Número mínimo de muestras requeridas en un nodo hoja

In [ ]:
# Definir los valores de los hiperparámetros a probar
max_depths = [3, 5, 7]
min_samples_splits = [2, 5, 10]
min_samples_leafs = [1, 3, 5]

# Crear listas para almacenar resultados
results = []

# Iterar sobre todas las combinaciones de hiperparámetros
for depth in max_depths:
    for split in min_samples_splits:
        for leaf in min_samples_leafs:
            # Crear y entrenar el modelo
            cart = DecisionTreeClassifier(
                max_depth=depth,
                min_samples_split=split,
                min_samples_leaf=leaf,
                random_state=42
            )
            cart.fit(train_X, train_y)
            
            # Calcular error en entrenamiento y validación
            train_pred = cart.predict(train_X)
            val_pred = cart.predict(val_X)
            
            train_error = 1 - accuracy_score(train_y, train_pred)
            val_error = 1 - accuracy_score(val_y, val_pred)
            
            # Guardar resultados
            results.append({
                'max_depth': depth,
                'min_samples_split': split,
                'min_samples_leaf': leaf,
                'train_error': train_error,
                'val_error': val_error
            })

# Convertir resultados a DataFrame para mejor visualización
results_df = pd.DataFrame(results)
print("\nTabla comparativa de resultados:")
print(results_df.to_string(index=False))

In [ ]:
# Visualizar los resultados
plt.figure(figsize=(12, 6))
sns.scatterplot(data=results_df, x='train_error', y='val_error', hue='max_depth', 
                size='min_samples_split', style='min_samples_leaf')
plt.title('Comparación de errores de entrenamiento vs validación')
plt.xlabel('Error de entrenamiento')
plt.ylabel('Error de validación')
plt.show()

# Encontrar la mejor combinación de hiperparámetros
best_model_idx = results_df['val_error'].idxmin()
best_params = results_df.iloc[best_model_idx]
print("\nMejor combinación de hiperparámetros:")
print(f"max_depth: {best_params['max_depth']}")
print(f"min_samples_split: {best_params['min_samples_split']}")
print(f"min_samples_leaf: {best_params['min_samples_leaf']}")
print(f"Error de validación: {best_params['val_error']:.4f}")

## Evaluación del mejor modelo en el conjunto de prueba

Se utilizara la mejor combinación de hiperparámetros encontrada para entrenar un modelo final y evaluarlo en el conjunto de prueba.

In [ ]:
# Entrenar el mejor modelo
best_cart = DecisionTreeClassifier(
    max_depth=best_params['max_depth'],
    min_samples_split=best_params['min_samples_split'],
    min_samples_leaf=best_params['min_samples_leaf'],
    random_state=42
)
best_cart.fit(train_X, train_y)

# Evaluar en el conjunto de prueba
test_pred = best_cart.predict(test_X)
test_error = 1 - accuracy_score(test_y, test_pred)

print("Resultados en el conjunto de prueba:")
print(f"Error de prueba: {test_error:.4f}")
print("\nReporte de clasificación detallado:")
print(classification_report(test_y, test_pred))

## Predicción con nuevos datos

Se creara un ejemplo inventado para probar el modelo en un nuevo caso.

In [ ]:
# Crear un nuevo ejemplo
nuevo_juego = pd.DataFrame({
    'title': ['New Adventure Game'],
    'platform': ['PS5'],
    'review_text': ['Amazing graphics and gameplay, highly recommended for all ages!'],
    'review_score': [9.5]
})

# Realizar la predicción
prediccion = best_cart.predict(nuevo_juego)
print("Predicción para el nuevo juego:")
print(f"Clasificación predicha: {prediccion[0]}")

## Conclusiones

1. **Pendiente**